# CS 431: Data-Intensive Distributed Computing (Fall 2026)
# Assignment 6: Graph Processing and Single-Source Shortest Paths

So far, you've counted words in product reviews (Assignment 1), built and analyzed a data warehouse from the [Steam platform](https://store.steampowered.com/) (Assignments 2 and 3), trained machine learning models (Assignment 4), and built and searched inverted indexes (Assignment 5).
This assignment is about graphs, and about one of the most basic questions you can ask of one: how far is every node from a given source?

For this assignment, you'll be using two road networks from the [9th DIMACS Implementation Challenge](http://www.diag.uniroma1.it/challenge9/download.shtml): New York City (`ny`) and the Northeastern United States (`ne`).
You'll build:

+ A degree analysis of both graphs in PySpark.
+ A parallel single-source shortest path (SSSP) algorithm in PySpark, one iteration at a time.
+ The same algorithm in DuckDB, where each iteration is a single SQL statement.

Along the way, you'll see what it takes to run an iterative graph algorithm on a dataflow engine, and you'll compare a scale-out engine (Spark) with a scale-up one (DuckDB) on the same problem.

You're expected to use a coding agent to help you with this assignment.
However, you are ultimately responsible for and accountable for _everything_, including design decisions, quality checks, bugs, edge cases, interpretation of the results, etc.
You are also responsible for _understanding_ what the coding agent is doing.

<font color="red"><b>Important Consideration</b></font>:
This assignment asks a number of short-answer questions.
Answer completely but succinctly.
We don't expect an overly verbose explanation: simply enough to convey your understanding of the material.
LLMs often use a lot of words but convey little actual content.

Your coding agent will likely be able to one-shot the answer, but you should resist that temptation because it will likely impede your understanding if you simply ask the LLM to give you the answer.
As a suggestion, avoid simply copying/pasting the question into the prompt or directly pointing your agent at a specific notebook cell.
Try to come up with the answer yourself, perhaps using the agent to explain related concepts.
In other words, use the agent to help you _understand_ the material, not write the answer for you.
Such a strategy might be more conducive to learning.

This assignment has two kinds of tasks:

+ In many places, we give you working code and ask you to explain it. Your answers should be about _this_ code and _your_ outputs (the numbers, files, and timings you see), not generic textbook descriptions.
+ In other places, you implement core functions to a precise specification. The checks we provide are not exhaustive: when grading, we'll also call your functions on inputs you haven't seen. So implement the specification, not just enough to pass the checks.

For each short answer, two or three clear sentences are usually enough. Refer to the displayed evidence rather than giving a general textbook explanation. Use your own measured values where requested; timings may differ between runs.

When doing this assignment:

+ Use PySpark RDDs (not DataFrames) in Sections 1 and 2. Do not read the graphs into Python data structures on the driver, except where provided code does so.
+ In Section 3, use DuckDB SQL. Each iteration of the algorithm must be exactly one SQL statement.
+ Use exactly the names, signatures, and output formats that we specify.
+ Keep the provided cells unchanged unless a `TODO` tells you otherwise.

To start, let's set up Spark and DuckDB. Run the following code and resolve any errors before continuing.

Plan for approximately **8 GB of RAM** for the complete notebook and about **1 GB of free disk space** for the graphs and the intermediate results.
These are planning estimates, not measured peak requirements for every machine.

Use the course Python environment. 
This assignment also needs DuckDB, which earlier assignments didn't use.
If DuckDB is missing, run:

```bash
pip install duckdb
```

As of the time the assignment was created, `pip` should pull in `duckdb` v2.0.1.

When Spark starts, it may print a few harmless warnings (for example, about native Hadoop libraries, incubator modules, or the Spark UI port); you can ignore them.

In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable

# TODO: Set SPARK_HOME for your machine if PySpark is not already installed.
# os.environ["SPARK_HOME"] = "/path/to/spark-4.2.0-bin-hadoop3"
if os.getenv("SPARK_HOME"):
    import findspark
    findspark.init()

import duckdb
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Assignment6").master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

# DuckDB gets the same number of threads as Spark has cores (local[4]), so the comparison is fair.
con = duckdb.connect()
con.execute("SET threads TO 4")

print("spark_version=" + spark.version)
print("duckdb_version=" + duckdb.__version__)

## 1. The Graphs

### 1.1 Get the Graphs

We'll be working with `graph-data.tar.gz` (29 MB), from `https://cs.uwaterloo.ca/~jimmylin/cs431-2026f/graph-data.tar.gz`.
The code cell below takes care of getting it for you: it downloads the archive into a `data/` directory next to this notebook, verifies its MD5 checksum, unpacks it, and verifies every file inside.

The archive contains two graphs, each in its own directory:

+ `ny`: the road network of New York City.
+ `ne`: the road network of the Northeastern United States.

Each graph has two files:

+ `vertices.csv` lists the node ids, one per line, after the header `id`.
+ `edges.csv` lists the directed, weighted edges, one per line, after the header `src,dst,weight`.
Nodes are road intersections, edges are road segments, and weights are integer segment lengths.

Here are the first three lines of `ny/edges.csv`:

```text
src,dst,weight
1,2,803
2,1,803
```

That is, you can get from node 1 to node 2 at a cost of 803.

In [ ]:
import hashlib
import shutil
import tarfile
import urllib.request

GRAPH_DATA_URL = "https://cs.uwaterloo.ca/~jimmylin/cs431-2026f/graph-data.tar.gz"
GRAPH_DATA_TGZ_MD5 = "122b471c1aad2eb7afd09be35bc16f7e"
GRAPH_FILE_MD5 = {
    "ny/vertices.csv": "b1bc1a2eaa07a2e0e70e5cda073bca75",
    "ny/edges.csv": "bdbf33b2a2b9dd999a06ec97baa77034",
    "ne/vertices.csv": "596983e07193d0094231c710c5ecaac5",
    "ne/edges.csv": "da2fa7ede9fcfacb81e0b212b8ec6301",
}

# The graphs go in data/graph-data/ next to this notebook. If you already have an unpacked copy
# elsewhere, point the environment variable CS431_A6_GRAPH_DATA at its graph-data directory instead.
GRAPH_DATA_TGZ_PATH = os.path.join("data", "graph-data.tar.gz")
GRAPH_DATA_DIR = os.getenv("CS431_A6_GRAPH_DATA", os.path.join("data", "graph-data"))

def md5sum(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def check_md5(path, expected):
    if md5sum(path) != expected:
        raise ValueError(f"{path} is corrupt or incomplete (wrong MD5 checksum): delete it and re-run this cell.")

if not all(os.path.exists(os.path.join(GRAPH_DATA_DIR, name)) for name in GRAPH_FILE_MD5):
    if not os.path.exists(GRAPH_DATA_TGZ_PATH):
        print(f"Downloading {GRAPH_DATA_URL} (29 MB)...")
        os.makedirs(os.path.dirname(GRAPH_DATA_TGZ_PATH), exist_ok=True)
        with urllib.request.urlopen(GRAPH_DATA_URL) as response, open(GRAPH_DATA_TGZ_PATH + ".part", "wb") as out:
            shutil.copyfileobj(response, out, 1 << 20)
        os.replace(GRAPH_DATA_TGZ_PATH + ".part", GRAPH_DATA_TGZ_PATH)
    check_md5(GRAPH_DATA_TGZ_PATH, GRAPH_DATA_TGZ_MD5)
    print(f"Unpacking into {os.path.dirname(GRAPH_DATA_DIR) or '.'}...")
    with tarfile.open(GRAPH_DATA_TGZ_PATH) as tar:
        tar.extractall(os.path.dirname(GRAPH_DATA_DIR) or ".", filter="data")

for name, expected in GRAPH_FILE_MD5.items():
    path = os.path.join(GRAPH_DATA_DIR, name)
    check_md5(path, expected)
    print(f"{name:<16} bytes={os.path.getsize(path)}")
print("checksum=ok")

You should see `checksum=ok`.
Don't continue until you do.

The cell only downloads and unpacks the first time; when you re-run it, it just re-verifies the checksums.
Do not commit the `data/` directory to your repository.

### 1.2 Loading the Graphs

The cell below loads the edges of each graph into an RDD of `(src, (dst, weight))` pairs, where all three values are integers, and counts the nodes and edges.
We key each edge by its source node because that's how the algorithms below use it: to follow the edges _out_ of a node.

Just run the cell.

In [ ]:
GRAPHS = ["ny", "ne"]

# Each edge line is "src,dst,weight".
def parse_edge(line):
    src, dst, weight = line.split(",")
    return int(src), (int(dst), int(weight))

def load_edges(path):
    return sc.textFile(path).filter(lambda line: not line.startswith("src")).map(parse_edge)

graph_edges = {}
vertex_n = {}
for graph in GRAPHS:
    graph_edges[graph] = load_edges(os.path.join(GRAPH_DATA_DIR, graph, "edges.csv")).cache()
    vertex_n[graph] = (
        sc.textFile(os.path.join(GRAPH_DATA_DIR, graph, "vertices.csv"))
        .filter(lambda line: not line.startswith("id"))
        .count()
    )
    print(f"graph={graph} vertex_n={vertex_n[graph]} edge_n={graph_edges[graph].count()}")

graph_edges["ny"].take(3)

Before proceeding, make sure you understand the structure of the data and what we just did above.

### 1.3 Degrees

The _out-degree_ of a node is the number of edges leaving it; its _in-degree_ is the number of edges arriving at it.
Count edge records: if the same `(src, dst)` pair appears twice in `edges.csv`, it adds 2 to the out-degree of `src` and 2 to the in-degree of `dst`.

Implement three functions:

+ `out_degrees(edges)`: `edges` is an RDD of `(src, (dst, weight))` pairs, like the ones loaded above. Return an RDD of `(node, out_degree)` pairs with exactly one record for each node that has at least one outgoing edge.
+ `in_degrees(edges)`: the same, for incoming edges: one `(node, in_degree)` record for each node that has at least one incoming edge.
+ `degree_histogram(degrees)`: `degrees` is an RDD of `(node, degree)` pairs, like the ones your functions above return. Return a Python list of `(degree, node_count)` tuples, sorted by degree in ascending order, where `node_count` is the number of nodes with that degree. Only degrees that occur appear in the list.

For example, for the edges `1→2`, `1→3`, `2→3`, the out-degrees are `{(1, 2), (2, 1)}`, the in-degrees are `{(2, 1), (3, 2)}`, and the histogram of the in-degrees is `[(1, 1), (2, 1)]`.
Node 3 has no outgoing edges, so it has no out-degree record (not a record with degree 0).

In [ ]:
## acell_613_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

def out_degrees(edges):
    pass

def in_degrees(edges):
    pass

def degree_histogram(degrees):
    pass

######### END STUDENT CODE #########

Let's check your functions on a tiny graph where we know the answer.
It has a repeated edge (`0→2` appears twice), a node with no incoming edges (5), and edges of different weights.
We'll use the same graph to check your shortest-path implementations later.

Just run the cell.

In [ ]:
from pyspark import RDD

TOY_EDGES = [
    (0, 1, 4), (0, 2, 1), (0, 2, 1), (2, 1, 1), (1, 3, 1),
    (2, 3, 5), (3, 4, 3), (4, 0, 1), (5, 0, 2),
]
toy_edges = sc.parallelize([(src, (dst, weight)) for src, dst, weight in TOY_EDGES], 3)

toy_out, toy_in = out_degrees(toy_edges), in_degrees(toy_edges)

assert isinstance(toy_out, RDD) and isinstance(toy_in, RDD), "out_degrees and in_degrees must return RDDs"
assert sorted(toy_out.collect()) == [(0, 3), (1, 1), (2, 2), (3, 1), (4, 1), (5, 1)]
assert sorted(toy_in.collect()) == [(0, 2), (1, 2), (2, 2), (3, 2), (4, 1)]
assert degree_histogram(toy_out) == [(1, 4), (2, 1), (3, 1)]
assert degree_histogram(toy_in) == [(1, 1), (2, 4)]
print("degree_checks=passed")

You should see `degree_checks=passed`.
If not, stop and debug.

Now let's run your functions on both road networks.
The cell below prints, for each graph, the mean in- and out-degree (over nodes with at least one such edge), the node with the largest in- and out-degree (the smallest id, if there's a tie), and the two histograms side by side.

Just run the cell.

In [ ]:
degree_hists = {}
for graph in GRAPHS:
    out_deg, in_deg = out_degrees(graph_edges[graph]).cache(), in_degrees(graph_edges[graph]).cache()
    in_hist, out_hist = degree_histogram(in_deg), degree_histogram(out_deg)
    degree_hists[graph] = (in_hist, out_hist)

    print(f"graph={graph}")
    print(f"mean_out_degree={out_deg.values().mean():.4f} (over {out_deg.count()} nodes)")
    print(f"mean_in_degree={in_deg.values().mean():.4f} (over {in_deg.count()} nodes)")
    for label, degrees in [("out", out_deg), ("in", in_deg)]:
        node, degree = degrees.takeOrdered(1, key=lambda kv: (-kv[1], kv[0]))[0]
        print(f"max_{label}_degree={degree} (node={node})")
    print(f"{'degree':>6} {'in_node_n':>10} {'out_node_n':>10}")
    in_counts, out_counts = dict(in_hist), dict(out_hist)
    for degree in sorted(in_counts.keys() | out_counts.keys()):
        print(f"{degree:>6} {in_counts.get(degree, 0):>10} {out_counts.get(degree, 0):>10}")
    print()
    out_deg.unpersist()
    in_deg.unpersist()

Let's plot the histograms. Note that the vertical axis is logarithmic.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(GRAPHS), figsize=(10, 3.5))
for ax, graph in zip(axes, GRAPHS):
    in_hist, out_hist = degree_hists[graph]
    ax.bar([d - 0.2 for d, _ in in_hist], [n for _, n in in_hist], width=0.4, label="in-degree")
    ax.bar([d + 0.2 for d, _ in out_hist], [n for _, n in out_hist], width=0.4, label="out-degree")
    ax.set_yscale("log")
    ax.set_xlabel("degree")
    ax.set_ylabel("number of nodes")
    ax.set_title(graph)
    ax.legend()
plt.tight_layout()
plt.show()

The in-degree and out-degree histograms are identical in both graphs.
The cell below takes a closer look at the edges themselves: it counts the edges `u→v` with weight `w` for which there is _no_ edge `v→u` with the same weight `w`, and the `(src, dst)` pairs that appear more than once.

Just run the cell.

In [ ]:
for graph in GRAPHS:
    arcs = graph_edges[graph].map(lambda kv: (kv[0], kv[1][0], kv[1][1]))
    without_reverse_n = arcs.subtract(arcs.map(lambda arc: (arc[1], arc[0], arc[2]))).count()
    repeated_pair_n = (
        arcs.map(lambda arc: ((arc[0], arc[1]), 1))
        .reduceByKey(lambda a, b: a + b)
        .filter(lambda kv: kv[1] > 1)
        .count()
    )
    print(f"graph={graph} edges_without_reverse_n={without_reverse_n} repeated_pair_n={repeated_pair_n}")

**Q1.3(a): Why are the in-degree and out-degree histograms identical?**

What property of these edge lists explains it?
Would identical histograms _by themselves_ be enough to conclude that a graph has this property? If yes, explain why; if not, give a small counterexample.

// acell_613_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

**Q1.3(b): What does the degree distribution mean for Spark?**

Compare these degree distributions with what you'd expect from a social network or the web graph, where a few nodes have millions of edges.
Why is the road-network distribution good news for Spark operations that group records by node, such as `reduceByKey` or `join`?
(Think back to the largest postings list in Assignment 5.)

// acell_613_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.

## 2. Single-Source Shortest Paths with Spark

### 2.1 Expanding the Frontier

Given a source node, we want the length of the shortest path from the source to every other node, where the length of a path is the sum of its edge weights.
On a single machine, you'd use Dijkstra's algorithm, which repeatedly settles the closest unsettled node using a global priority queue.
That's inherently sequential: one node at a time.

Instead, we'll use the parallel breadth-first search from the week 12 reading (_Data-Intensive Text Processing with MapReduce_, Chapter 5), generalized to weighted edges.
We keep a set of _reachable_ nodes, each with the shortest distance found so far, as `(node, dist)` pairs.
Initially, only the source is reachable, at distance 0.
Each iteration expands the frontier by one hop:

+ every reachable node `u` with distance `d` offers the distance `d + w` to each node `v` it has an edge `u→v` with weight `w` to;
+ every node keeps the minimum of its current distance (if it has one) and all the distances offered to it.

After `k` iterations, the reachable set contains exactly the nodes that can be reached from the source in at most `k` edges, and each node's distance is the length of the shortest path _that uses at most `k` edges_.
If we keep iterating until nothing changes, the distances are the true shortest-path distances.

Unlike the MapReduce version in the reading, which passes each node's adjacency list along with its distance, we keep the graph structure in a separate RDD and join the reachable set with it in each iteration.

Implement `expand_frontier(reachable, edges)`:

+ `reachable` is an RDD of `(node, dist)` pairs, with exactly one record per node.
+ `edges` is an RDD of `(src, (dst, weight))` pairs, like the ones loaded in Section 1.2.
+ Return an RDD of `(node, dist)` pairs: the reachable set after one more iteration, as defined above, with exactly one record per node.
+ Use RDD transformations only: don't collect `reachable` or `edges` to the driver.
+ Any partitioning of the result is fine: the driver repartitions it before the next iteration.

For example, with the edges `1→2` (weight 5), `1→3` (weight 1), and `3→2` (weight 1), and `reachable` containing `(1, 0)`, one iteration returns `{(1, 0), (2, 5), (3, 1)}`. A second iteration returns `{(1, 0), (2, 2), (3, 1)}`: node 2's distance went _down_, because a path with more edges turned out to be shorter.

In [ ]:
## acell_621_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

def expand_frontier(reachable, edges):
    pass

######### END STUDENT CODE #########

Let's check your function on the toy graph from Section 1.3, starting from node 0.
The expected distances change in later iterations: for example, node 1 is first reached at distance 4 (`0→1`), but in the next iteration at distance 2 (`0→2→1`).

Just run the cell.

In [ ]:
# Expected reachable sets from node 0 after 1, 2, ..., 5 iterations.
TOY_EXPECTED = [
    {0: 0, 1: 4, 2: 1},
    {0: 0, 1: 2, 2: 1, 3: 5},
    {0: 0, 1: 2, 2: 1, 3: 3, 4: 8},
    {0: 0, 1: 2, 2: 1, 3: 3, 4: 6},
    {0: 0, 1: 2, 2: 1, 3: 3, 4: 6},  # No change: the algorithm has converged.
]

toy_reachable = sc.parallelize([(0, 0)], 2)
for iteration, expected in enumerate(TOY_EXPECTED, start=1):
    toy_reachable = expand_frontier(toy_reachable, toy_edges)
    assert isinstance(toy_reachable, RDD), "expand_frontier must return an RDD"
    actual = toy_reachable.collect()
    assert len(actual) == len(dict(actual)), f"iteration {iteration}: expected exactly one record per node"
    assert dict(actual) == expected, f"iteration {iteration}: expected {expected}, got {dict(actual)}"

# Node 5 has an outgoing edge but no incoming ones.
assert dict(expand_frontier(sc.parallelize([(5, 0)]), toy_edges).collect()) == {5: 0, 0: 2}
# A distance is never replaced by a longer one.
assert dict(expand_frontier(sc.parallelize([(0, 0), (1, 1)]), toy_edges).collect()) == {0: 0, 1: 1, 2: 1, 3: 2}
print("expand_frontier_checks=passed")

You should see `expand_frontier_checks=passed`.
If not, stop and debug.

### 2.2 Running on the Road Networks

The cell below defines `run_sssp_spark`, the driver loop that calls your `expand_frontier`.
It doesn't keep the reachable set in memory from one iteration to the next.
Instead, it writes each iteration's reachable set to disk as a directory of `node,dist` text files, `reachable/<graph>-src-<source>-iterNNN`, and reads it back in the next iteration.
Iteration 0 is a single file holding `<source>,0`.

The driver hash-partitions the edges once and caches them, then partitions the reachable set the same way before each join.

Just run the cell.

In [ ]:
import glob
import time
from pyspark.rdd import portable_hash

REACHABLE_DIR = "reachable"
NUM_PARTITIONS = 8

partitioned_edges = {graph: graph_edges[graph].partitionBy(NUM_PARTITIONS, portable_hash).cache() for graph in GRAPHS}

def parse_reachable(line):
    node, dist = line.split(",")
    return int(node), int(dist)

def run_sssp_spark(graph, source, iterations, edges=None, copartition=True, name=None):
    # Returns the path prefix of the iteration directories and the wall time of each iteration.
    edges = partitioned_edges[graph] if edges is None else edges
    prefix = os.path.join(REACHABLE_DIR, name or f"{graph}-src-{source}")
    for path in glob.glob(prefix + "-iter*"):
        shutil.rmtree(path)
    os.makedirs(f"{prefix}-iter000")
    with open(os.path.join(f"{prefix}-iter000", "source.csv"), "w") as f:
        f.write(f"{source},0\n")

    iteration_s = []
    for i in range(1, iterations + 1):
        start = time.perf_counter()
        reachable = sc.textFile(f"{prefix}-iter{i - 1:03}").map(parse_reachable)
        if copartition:
            reachable = reachable.partitionBy(NUM_PARTITIONS, portable_hash)
        expanded = expand_frontier(reachable, edges)
        expanded.map(lambda kv: f"{kv[0]},{kv[1]}").saveAsTextFile(f"{prefix}-iter{i:03}")
        iteration_s.append(time.perf_counter() - start)
    return prefix, iteration_s

def summarize(pairs):
    # Sort by distance, then node id; fingerprint the result as "node,dist" lines.
    rows = sorted(pairs, key=lambda kv: (kv[1], kv[0]))
    text = "".join(f"{node},{dist}\n" for node, dist in rows)
    return {
        "reachable_n": len(rows),
        "max_dist": rows[-1][1],
        "dist_sum": sum(dist for _, dist in rows),
        "result_md5": hashlib.md5(text.encode()).hexdigest(),
    }, rows

We run 20 iterations from two sources (nodes 6 and 1024) on each graph.
For each run, the cell prints the number of reachable nodes, the largest and total distance, how many nodes changed (were newly reached or got a shorter distance) in the final iteration, a fingerprint of the result, and the closest and farthest nodes.

Just run the cell. It may take a few minutes.

In [ ]:
ITERATIONS = 20
CONFIGS = [("ny", 6), ("ne", 6), ("ny", 1024), ("ne", 1024)]

spark_results, spark_iteration_s = {}, {}
for graph, source in CONFIGS:
    prefix, spark_iteration_s[graph, source] = run_sssp_spark(graph, source, ITERATIONS)
    last = sc.textFile(f"{prefix}-iter{ITERATIONS:03}").map(parse_reachable)
    previous = sc.textFile(f"{prefix}-iter{ITERATIONS - 1:03}").map(parse_reachable)
    changed_n = last.leftOuterJoin(previous).filter(lambda kv: kv[1][1] is None or kv[1][0] < kv[1][1]).count()

    summary, rows = summarize(last.collect())
    assert summary["reachable_n"] == len(dict(rows)), "expected exactly one record per node"
    spark_results[graph, source] = rows
    print(f"graph={graph} source={source} iterations={ITERATIONS} changed_in_last_iteration_n={changed_n}")
    print(" ".join(f"{key}={value}" for key, value in summary.items()))
    print(f"closest={rows[:5]} farthest={rows[-3:]}")
    print(f"wall_s={sum(spark_iteration_s[graph, source]):.1f}")
    print()

Look at a few of the output files under `reachable/` to make sure you understand what the driver wrote.

The next cell shows what would happen if the driver kept the reachable set in memory instead: it chains `k` iterations of `expand_frontier` without writing anything, then counts the result.
For each `k`, it prints the number of lines in Spark's description of the RDD's lineage (`toDebugString`) and the time to count it.

Just run the cell.

In [ ]:
for k in [1, 4, 8, 12]:
    chained = sc.parallelize([(6, 0)])
    for _ in range(k):
        chained = expand_frontier(chained.partitionBy(NUM_PARTITIONS, portable_hash), partitioned_edges["ny"])
    start = time.perf_counter()
    count = chained.count()
    count_s = time.perf_counter() - start
    try:
        lineage_lines = len(chained.toDebugString().decode().splitlines())
    except Exception as e:  # A deep enough lineage can't even be printed.
        lineage_lines = f"failed ({type(e).__name__})"
    print(f"chained_iterations={k:>2} lineage_lines={lineage_lines:>6} reachable_n={count:>4} count_s={count_s:.1f}")

**Q2.2(a): Why write every iteration to disk?**

Writing and re-reading every iteration looks wasteful.
Based on the lineage output above, what would happen to the lineage if we ran the 1,000+ iterations needed for the distances to converge (see Section 3.3) without writing to disk?
Why does that matter, both for planning the job and for recovering from a failed task?

// acell_622_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

The last experiment runs three iterations from node 6 on `ne` twice: once as above, and once with the original, unpartitioned edges and without partitioning the reachable set.
It prints the median time per iteration of each run.

Just run the cell.

In [ ]:
import statistics

for label, edges, copartition in [
    ("copartitioned", partitioned_edges["ne"], True),
    ("unpartitioned", graph_edges["ne"], False),
]:
    _, seconds = run_sssp_spark("ne", 6, 3, edges=edges, copartition=copartition, name=f"ne-src-6-{label}")
    print(f"{label:>14} median_iteration_s={statistics.median(seconds):.2f}")

**Q2.2(b): What does co-partitioning save?**

In PySpark, `join` is implemented as a `union` of both inputs followed by a `groupByKey`.
`groupByKey` skips the shuffle if its input is already hash-partitioned by key into the same partitions it would produce.
Using this and the timings above, explain what is shuffled in each iteration in the two runs, and why only one of them pays a cost proportional to the size of the graph.

// acell_622_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.

## 3. Single-Source Shortest Paths with DuckDB

### 3.1 One SQL Statement per Iteration

Now let's run the same algorithm on DuckDB, an in-process analytical database that runs on a single machine.
The cell below loads each graph's edges into separate DuckDB tables, `edges_ny` and `edges_ne`, with the integer columns `src`, `dst`, and `weight`.

Just run the cell.

In [ ]:
for graph in GRAPHS:
    path = os.path.join(GRAPH_DATA_DIR, graph, "edges.csv")
    con.execute(f'''
        CREATE OR REPLACE TABLE edges_{graph} AS
        SELECT * FROM read_csv('{path}', header = true,
                               columns = {{'src': 'INTEGER', 'dst': 'INTEGER', 'weight': 'INTEGER'}})
    ''')
    print(f"graph={graph} edge_n={con.execute(f'SELECT count(*) FROM edges_{graph}').fetchone()[0]}")

In DuckDB, the reachable set after each iteration is a table with the columns `node` (`INTEGER`) and `dist` (`BIGINT`), one row per node.

Implement `expand_frontier_sql(table_in, table_out, edges_table)`:

+ `table_in` is the name of an existing reachable table; `edges_table` is the name of an existing edges table, with the columns `src`, `dst`, and `weight`.
+ Return a string containing exactly one SQL statement that creates a new table named `table_out` (which doesn't exist yet) with the columns `node` and `dist`: the reachable set after one more iteration, as defined in Section 2.1. The column types follow from the input tables, so you don't need casts. Whitespace around the statement is fine, but don't add a second statement.
+ Don't execute the statement: the provided driver does that.
+ Don't modify or drop `table_in` or `edges_table`.

For example, `expand_frontier_sql("reachable003", "reachable004", "edges_ny")` returns a statement that starts with `CREATE TABLE reachable004 AS ...`.

In [ ]:
## acell_631_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

def expand_frontier_sql(table_in, table_out, edges_table):
    return f'''
    -- SQL statement here
    '''

######### END STUDENT CODE #########

Let's check your function on the toy graph, with the same expected results as in Section 2.1.

Just run the cell.

In [ ]:
def create_reachable(table, rows):
    con.execute(f"DROP TABLE IF EXISTS {table}")
    con.execute(f"CREATE TABLE {table} (node INTEGER, dist BIGINT)")
    con.executemany(f"INSERT INTO {table} VALUES (?, ?)", rows)

def run_statement(sql):
    # Exactly one statement, and it creates a table.
    assert isinstance(sql, str), "expand_frontier_sql must return a string"
    statements = con.extract_statements(sql)
    assert len(statements) == 1, f"expected exactly one SQL statement, got {len(statements)}"
    assert statements[0].type == duckdb.StatementType.CREATE, "expected a CREATE TABLE statement"
    con.execute(sql)

con.execute("CREATE OR REPLACE TABLE toy_edges (src INTEGER, dst INTEGER, weight INTEGER)")
con.executemany("INSERT INTO toy_edges VALUES (?, ?, ?)", TOY_EDGES)

create_reachable("toy_reachable000", [(0, 0)])
for iteration, expected in enumerate(TOY_EXPECTED, start=1):
    table_in, table_out = f"toy_reachable{iteration - 1:03}", f"toy_reachable{iteration:03}"
    con.execute(f"DROP TABLE IF EXISTS {table_out}")
    run_statement(expand_frontier_sql(table_in, table_out, "toy_edges"))
    actual = con.execute(f"SELECT node, dist FROM {table_out}").fetchall()
    assert len(actual) == len(dict(actual)), f"iteration {iteration}: expected exactly one row per node"
    assert dict(actual) == expected, f"iteration {iteration}: expected {expected}, got {dict(actual)}"
assert con.execute("SELECT * FROM toy_reachable000").fetchall() == [(0, 0)], "table_in must not be modified"

create_reachable("toy_from_5", [(5, 0)])
con.execute("DROP TABLE IF EXISTS toy_from_5_next")
run_statement(expand_frontier_sql("toy_from_5", "toy_from_5_next", "toy_edges"))
assert dict(con.execute("SELECT node, dist FROM toy_from_5_next").fetchall()) == {5: 0, 0: 2}
print("expand_frontier_sql_checks=passed")

You should see `expand_frontier_sql_checks=passed`.
If not, stop and debug.

The cell below defines `run_sssp_duckdb`, the DuckDB counterpart of `run_sssp_spark`, and runs the same four configurations.
It keeps every iteration in a table named `reachableNNN` (dropping tables it no longer needs), and checks that DuckDB and Spark computed exactly the same results.

Just run the cell.

In [ ]:
def drop_reachable_tables():
    for (table,) in con.execute("SELECT table_name FROM duckdb_tables() WHERE table_name LIKE 'reachable%'").fetchall():
        con.execute(f"DROP TABLE {table}")

def changed_count(table_in, table_out):
    # Nodes that were newly reached or got a shorter distance.
    return con.execute(f'''
        SELECT count(*) FROM {table_out} AS o LEFT JOIN {table_in} AS i USING (node)
        WHERE i.dist IS NULL OR o.dist < i.dist
    ''').fetchone()[0]

def run_sssp_duckdb(graph, source, iterations):
    # Returns the name of the final table and the wall time of each iteration.
    drop_reachable_tables()
    create_reachable("reachable000", [(source, 0)])
    iteration_s = []
    for i in range(1, iterations + 1):
        table_in, table_out = f"reachable{i - 1:03}", f"reachable{i:03}"
        start = time.perf_counter()
        run_statement(expand_frontier_sql(table_in, table_out, f"edges_{graph}"))
        iteration_s.append(time.perf_counter() - start)
        if i >= 2:
            con.execute(f"DROP TABLE reachable{i - 2:03}")
    return f"reachable{iterations:03}", iteration_s

duckdb_results, duckdb_iteration_s = {}, {}
for graph, source in CONFIGS:
    table, duckdb_iteration_s[graph, source] = run_sssp_duckdb(graph, source, ITERATIONS)
    changed_n = changed_count(f"reachable{ITERATIONS - 1:03}", table)
    summary, rows = summarize(con.execute(f"SELECT node, dist FROM {table}").fetchall())
    duckdb_results[graph, source] = rows
    print(f"graph={graph} source={source} iterations={ITERATIONS} changed_in_last_iteration_n={changed_n}")
    print(" ".join(f"{key}={value}" for key, value in summary.items()))
    print(f"wall_s={sum(duckdb_iteration_s[graph, source]):.2f}")
    print()

mismatched = [config for config in CONFIGS if duckdb_results[config] != spark_results[config]]
print(f"mismatched_config_n={len(mismatched)}")

`mismatched_config_n` must be zero.
If not, stop and debug.

### 3.2 Spark versus DuckDB

The cell below puts the per-iteration times of the two implementations side by side.
Both use four cores on the same machine.

Just run the cell.

In [ ]:
import pandas as pd

timing = pd.DataFrame([
    {
        "graph": graph,
        "source": source,
        "spark_total_s": sum(spark_iteration_s[graph, source]),
        "spark_median_iteration_s": statistics.median(spark_iteration_s[graph, source]),
        "duckdb_total_s": sum(duckdb_iteration_s[graph, source]),
        "duckdb_median_iteration_s": statistics.median(duckdb_iteration_s[graph, source]),
    }
    for graph, source in CONFIGS
])
timing["spark_divided_by_duckdb"] = timing["spark_total_s"] / timing["duckdb_total_s"]
print(timing.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

**Q3.2(a): Scale up or scale out?**

Which implementation is faster, and by roughly how much?
Give two reasons for the difference that come from how the two engines execute _this_ workload.
Then, drawing on the week 12 reading ("Scale Up or Scale Out for Graph Processing?"), describe when Spark would nevertheless be the right choice.

// acell_632_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

### 3.3 How Many Iterations?

Twenty iterations reach only a tiny fraction of each graph.
The cell below uses your `expand_frontier_sql` to keep iterating from node 6 until no node changes, on both graphs, and prints the progress at a few checkpoints: how many nodes are reachable and how many changed in that iteration.

Just run the cell. It takes a few minutes, and on `ne` it prints nothing for a minute or two between checkpoints: that's expected.

In [ ]:
CHECKPOINTS = {1, 10, 20, 50, 100, 200, 500, 1000, 1500, 2000}

for graph in GRAPHS:
    vertex_total = vertex_n[graph]
    drop_reachable_tables()
    create_reachable("reachable0000", [(6, 0)])
    start, i = time.perf_counter(), 0
    print(f"graph={graph} source=6")
    print(f"{'iteration':>9} {'reachable_n':>12} {'changed_n':>10}")
    while True:
        i += 1
        table_in, table_out = f"reachable{i - 1:04}", f"reachable{i:04}"
        run_statement(expand_frontier_sql(table_in, table_out, f"edges_{graph}"))
        changed_n = changed_count(table_in, table_out)
        con.execute(f"DROP TABLE {table_in}")
        if i in CHECKPOINTS or changed_n == 0:
            reachable_n = con.execute(f"SELECT count(*) FROM {table_out}").fetchone()[0]
            print(f"{i:>9} {reachable_n:>12} {changed_n:>10}")
        if changed_n == 0:
            break
    max_dist = con.execute(f"SELECT max(dist) FROM {table_out}").fetchone()[0]
    print(f"converged_iteration={i} reachable_fraction={reachable_n / vertex_total:.4f} max_dist={max_dist} wall_s={time.perf_counter() - start:.1f}")
    print()
drop_reachable_tables()

**Q3.3(a): Why does it take so many iterations?**

How many iterations did it take for the distances from node 6 to stop changing on each graph?
What property of the graph and the source determines that number?
Why is this algorithm a poor fit for road networks, compared with a social network?

// acell_633_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

**Q3.3(b): Can we do less work per iteration?**

Each iteration of `expand_frontier_sql` re-offers distances from _every_ reachable node, but the `changed_n` column shows that late iterations change only a small fraction of them.
Describe a change to the algorithm that does less work per iteration but computes exactly the same distances, and explain why the results are unchanged.

// acell_633_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.

All done!
Stop Spark and close the DuckDB connection.

In [ ]:
spark.stop()
con.close()

## 4. Reflection on AI Collaboration

Complete this log honestly. This is primarily for your own benefit; we'll read it, but it won't be graded.

+ What I asked the coding agent to do.
+ What worked and what needed correction.
+ One function I wrote (or rewrote) myself, and how I convinced myself it was correct beyond the provided checks.
+ One place where my lecture knowledge mattered.
+ One prompt I would write differently next time.

// acell_640_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

## 5. Assignment Submission

Before submitting:

+ Restart the kernel and run all cells from top to bottom.
+ Keep every `acell_...` identifier and every instructor-provided output statement unchanged.
+ Keep every required function, dictionary, and variable with its specified name, signature, and format.
+ Remove temporary debugging output and scratch cells, but retain the required outputs and written answers.
+ Do not commit the `data/` directory or the `reachable/` directory to your repository.
+ No standalone Python scripts or intermediate result files are required.
+ Make sure your notebook contains execution traces so that we can evaluate your implementation, for example, the degree statistics, the shortest-path results and fingerprints, the timings, outputs supporting your written answers, etc.

For submission details, see [the course homepage](https://lintool.github.io/cs431-2026f/assignments/assignment6.html).
Make sure you conform to the requirements articulated there.